# EC523 Project — Speech Denoising

Tamerlan Baimurat, Punnisa Amornsirikul, Jiaxing Wang, Michael Lwe

{baimurat, punnisa, jiaxingw, mlwe}@bu.edu

Spring 2026


**Dataset:** 23,075 training + 824 test paired clean/noisy utterances (16 kHz, mono).
All audio has been preprocessed to complex STFT tensors (`complex64`, shape `[257, T]`) and served via a public HTTP endpoint: https://ec523.tamerlanbaimurat.com.

**Goal:** Build and train a neural network that takes a noisy STFT spectrogram as input and produces a denoised (clean) spectrogram as output.

# 1. Setup

This section loads all required libaries for our project.

In [1]:
!pip install flash-linear-attention

In [2]:
from __future__ import print_function
import io
import json
import os
import random
import threading
import time
from urllib.parse import quote

import matplotlib.pyplot as plt
import numpy as np
import requests
from requests.adapters import HTTPAdapter
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from fla.layers import DeltaNet

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [4]:
def stft_to_mag_phase(stft_complex):
    mag = stft_complex.abs()
    phase = stft_complex.angle()
    return mag, phase

# 2. Retrieving Data

The following Python script retrives the training and testing dataset from our HTML endpoint, https://ec523.tamerlanbaimurat.com.

In [5]:
BASE_URL = "https://ec523.tamerlanbaimurat.com"
PREFIX   = "ec523project"
MAX_RETRIES = 5

def public_url(key: str) -> str:
    return f"{BASE_URL}/{quote(key.lstrip('/'), safe='/')}"

_local = threading.local()

def _session() -> requests.Session:
    if not hasattr(_local, "s"):
        s = requests.Session()
        adapter = HTTPAdapter(pool_connections=64, pool_maxsize=64)
        s.mount("https://", adapter)
        _local.s = s
    return _local.s

def fetch_bytes(key: str, timeout: int = 60) -> bytes:
    for attempt in range(MAX_RETRIES):
        try:
            r = _session().get(public_url(key), timeout=timeout)
            if r.status_code == 503:
                raise requests.exceptions.ConnectionError("503")
            r.raise_for_status()
            return r.content
        except (requests.exceptions.ConnectionError,
                requests.exceptions.Timeout,
                requests.exceptions.HTTPError):
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(2 ** attempt + random.random())
    return b""

def fetch_manifest(split: str) -> list[dict]:
    key = f"{PREFIX}/manifests/{split}.jsonl"
    text = fetch_bytes(key).decode()
    return [json.loads(line) for line in text.splitlines() if line.strip()]


class AdaptivePool:
    """AIMD concurrency: starts fast, backs off on 503, recovers quickly."""

    def __init__(self, initial=32, minimum=4, maximum=64, grow_every=20):
        self._sem_value = initial
        self._sem = threading.Semaphore(initial)
        self._lock = threading.Lock()
        self._min = minimum
        self._max = maximum
        self._grow_every = grow_every
        self.successes = 0
        self.errors = 0

    @property
    def window(self):
        return self._sem_value

    def acquire(self):
        self._sem.acquire()

    def release_success(self):
        with self._lock:
            self.successes += 1
            if self.successes % self._grow_every == 0 and self._sem_value < self._max:
                self._sem_value += 1
                self._sem.release()
                return
        self._sem.release()

    def release_error(self):
        with self._lock:
            self.errors += 1
            new = max(self._sem_value // 2, self._min)
            while self._sem_value > new:
                self._sem.acquire(blocking=False) or None
                self._sem_value -= 1
        self._sem.release()


class STFTDataset(Dataset):
    """Fetches clean/noisy STFT pairs from HTTP, with in-memory caching."""

    def __init__(self, records: list[dict], split: str):
        self.records = records
        self.split = split
        self._cache: dict[int, tuple[torch.Tensor, torch.Tensor]] = {}

    @classmethod
    def from_manifest(cls, split: str) -> "STFTDataset":
        records = fetch_manifest(split)
        print(f"[{split}] manifest loaded: {len(records)} pairs")
        return cls(records, split)

    def prefetch_all(self):
        """Download every pair into RAM with adaptive concurrency."""
        from concurrent.futures import ThreadPoolExecutor, as_completed

        to_fetch = [i for i in range(len(self.records)) if i not in self._cache]
        if not to_fetch:
            print(f"[{self.split}] all {len(self.records)} pairs already cached")
            return

        pool = AdaptivePool(initial=32, minimum=4, maximum=128)

        def _download(idx):
            pool.acquire()
            try:
                rec = self.records[idx]
                noisy_bytes = fetch_bytes(rec["noisy_stft_key"])
                clean_bytes = fetch_bytes(rec["clean_stft_key"])
                noisy = torch.load(io.BytesIO(noisy_bytes), map_location="cpu",
                                   weights_only=True)["stft"]
                clean = torch.load(io.BytesIO(clean_bytes), map_location="cpu",
                                   weights_only=True)["stft"]
                pool.release_success()
                return idx, noisy, clean
            except Exception:
                pool.release_error()
                raise

        print(f"[{self.split}] prefetching {len(to_fetch)} pairs "
              f"(adaptive window: {pool.window}→{pool._max}) …")
        t0 = time.perf_counter()
        done, errors = 0, 0

        # Use a large thread pool; AdaptivePool's semaphore controls actual concurrency
        with ThreadPoolExecutor(max_workers=64) as ex:
            futures = {ex.submit(_download, i): i for i in to_fetch}
            for fut in as_completed(futures):
                try:
                    idx, noisy, clean = fut.result()
                    self._cache[idx] = (noisy, clean)
                    done += 1
                except Exception as e:
                    errors += 1
                    if errors <= 3:
                        print(f"  ERROR idx={futures[fut]}: {e}")
                total = done + errors
                if total % 500 == 0 or total == len(to_fetch):
                    elapsed = time.perf_counter() - t0
                    print(f"  {total}/{len(to_fetch)}  {elapsed:.0f}s  "
                          f"{done/max(elapsed,1):.0f} pairs/s  "
                          f"window={pool.window}  errors={errors}")
        elapsed = time.perf_counter() - t0
        print(f"[{self.split}] prefetch done: {done} OK, {errors} errors, "
              f"{elapsed:.0f}s")

        # Sequential retry for any failures
        remaining = [i for i in to_fetch if i not in self._cache]
        if remaining:
            print(f"[{self.split}] retrying {len(remaining)} failures …")
            for idx in remaining:
                try:
                    _, noisy, clean = _download(idx)
                    self._cache[idx] = (noisy, clean)
                except Exception as e:
                    print(f"  SKIP idx={idx}: {e}")

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int) -> dict:
        rec = self.records[idx]

        if idx in self._cache:
            noisy, clean = self._cache[idx]
        else:
            noisy = torch.load(io.BytesIO(fetch_bytes(rec["noisy_stft_key"])),
                               map_location="cpu", weights_only=True)["stft"]
            clean = torch.load(io.BytesIO(fetch_bytes(rec["clean_stft_key"])),
                               map_location="cpu", weights_only=True)["stft"]
            self._cache[idx] = (noisy, clean)

        return {
            "pair_id":     rec["pair_id"],
            "split":       self.split,
            "sample_rate": int(rec["sample_rate"]),
            "noisy":       noisy,          # complex64 [257, T]
            "clean":       clean,          # complex64 [257, T]
            "length":      noisy.shape[-1],
        }

In [6]:
# Fetch Testing Data from CloudFlare
test_ds  = STFTDataset.from_manifest("test")
test_ds.prefetch_all()

[test] manifest loaded: 824 pairs
[test] prefetching 824 pairs (adaptive window: 32→128) …
  500/824  9s  54 pairs/s  window=57  errors=0
  824/824  15s  55 pairs/s  window=73  errors=0
[test] prefetch done: 824 OK, 0 errors, 15s


# 3. Model Architecture & Load Model Parameters

This section details the code for the architecture of our model. [Expand on this later]

Input: `noisy_mag [B, 1, 257, T]` (log-magnitude spectrogram)

Output: predicted clean magnitude (same shape), or a mask in `[0, 1]`

STFT parameters: `n_fft=512`, `hop_length=160`, `win_length=400`, `window=hann`, 257 frequency bins.

In [7]:
from torch.utils.checkpoint import checkpoint as grad_checkpoint

# ---------------------------------------------------------------------------
# Building blocks
# ---------------------------------------------------------------------------

class ConvBlock(nn.Module):
    """Conv2d -> BatchNorm -> PReLU. Optionally downsamples frequency via stride."""
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=(1, 1), padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride=stride, padding=padding),
            nn.BatchNorm2d(out_ch),
            nn.PReLU(out_ch),
        )

    def forward(self, x):
        return self.block(x)


class DeconvBlock(nn.Module):
    """ConvTranspose2d -> BatchNorm -> PReLU. Upsamples frequency via stride."""
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=(2, 1),
                 padding=1, output_padding=(1, 0)):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, kernel_size, stride=stride,
                               padding=padding, output_padding=output_padding),
            nn.BatchNorm2d(out_ch),
            nn.PReLU(out_ch),
        )

    def forward(self, x):
        return self.block(x)


# ---------------------------------------------------------------------------
# Attention blocks
# ---------------------------------------------------------------------------

class DeltaNetBlock(nn.Module):
    """Pre-norm DeltaNet attention + FFN with residual connections."""
    def __init__(self, d_model, num_heads, ffn_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = DeltaNet(
            d_model=d_model,
            num_heads=num_heads,
            use_short_conv=True,
            conv_size=4,
            use_beta=True,
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * ffn_ratio),
            nn.GELU(),
            nn.Linear(d_model * ffn_ratio, d_model),
        )

    def forward(self, x):
        h = self.norm1(x)
        h, *_ = self.attn(h)
        x = x + h
        x = x + self.ffn(self.norm2(x))
        return x


class FullAttentionBlock(nn.Module):
    """Pre-norm self-attention using Flash Attention via scaled_dot_product_attention."""
    def __init__(self, d_model, num_heads, ffn_ratio=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.norm1 = nn.LayerNorm(d_model)
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * ffn_ratio),
            nn.GELU(),
            nn.Linear(d_model * ffn_ratio, d_model),
        )

    def forward(self, x):
        B, T, D = x.shape
        h = self.norm1(x)
        qkv = self.qkv(h).reshape(B, T, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)  # each [B, H, T, D_h]
        h = F.scaled_dot_product_attention(q, k, v)       # Flash Attention, O(T) memory
        h = h.transpose(1, 2).reshape(B, T, D)
        h = self.out_proj(h)
        x = x + h
        x = x + self.ffn(self.norm2(x))
        return x


# ---------------------------------------------------------------------------
# Main model
# ---------------------------------------------------------------------------

FREQ_CHUNK = 13  # process 13 freq bands at a time (65 / 13 = 5 chunks)

class SpeechDenoiser(nn.Module):
    """
    CNN encoder-decoder with DeltaNet + Full Attention bottleneck.

    Outputs:
        mask      [B, 1, 257, T]  sigmoid mask in [0, 1]
        cls_logit [B, 1]          raw logit for noisy/clean classification
    """

    def __init__(self, channels=64, n_heads=8, n_attn_layers=6, ffn_ratio=4):
        super().__init__()
        C = channels
        bottleneck_dim = C * 4   # 256

        # --- Encoder ---
        self.enc1 = ConvBlock(1, C)                                  # -> [B, 64, 257, T]
        self.enc2 = ConvBlock(C, C * 2, stride=(2, 1))              # -> [B, 128, 129, T]
        self.enc3 = ConvBlock(C * 2, bottleneck_dim, stride=(2, 1)) # -> [B, 256, 65, T]

        # --- Attention bottleneck ---
        self.attn_layers = nn.ModuleList()
        for i in range(n_attn_layers):
            if i % 2 == 0:
                self.attn_layers.append(
                    DeltaNetBlock(bottleneck_dim, n_heads, ffn_ratio))
            else:
                self.attn_layers.append(
                    FullAttentionBlock(bottleneck_dim, n_heads, ffn_ratio))
        self.attn_norm = nn.LayerNorm(bottleneck_dim)

        # --- Decoder (with skip connections) ---
        self.dec3 = DeconvBlock(bottleneck_dim, C * 2, stride=(2, 1),
                                padding=1, output_padding=(1, 0))
        self.dec2 = DeconvBlock(C * 4, C, stride=(2, 1),
                                padding=1, output_padding=(0, 0))
        self.dec1 = nn.Sequential(
            nn.Conv2d(C * 2, 1, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )

        # --- Classification head ---
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(bottleneck_dim, 1),
        )

    def _run_attention(self, h):
        """Run attention layers on a [N, T, C] tensor with gradient checkpointing."""
        for layer in self.attn_layers:
            h = grad_checkpoint(layer, h, use_reentrant=False)
        return self.attn_norm(h)

    def forward(self, x):
        # x: [B, 1, 257, T]

        # Encoder
        e1 = self.enc1(x)    # [B, C, 257, T]
        e2 = self.enc2(e1)   # [B, 2C, 129, T]
        e3 = self.enc3(e2)   # [B, 4C, 65, T]

        # Attention bottleneck: process freq bands in chunks to limit memory
        B, C_bn, F, T = e3.shape
        h_4d = e3.permute(0, 2, 3, 1)  # [B, F, T, C]

        chunks_out = []
        for i in range(0, F, FREQ_CHUNK):
            chunk = h_4d[:, i:i + FREQ_CHUNK]                    # [B, chunk_f, T, C]
            cf = chunk.shape[1]
            chunk = chunk.reshape(B * cf, T, C_bn)               # [B*chunk_f, T, C]
            chunk = self._run_attention(chunk)
            chunks_out.append(chunk.reshape(B, cf, T, C_bn))

        h = torch.cat(chunks_out, dim=1).permute(0, 3, 1, 2)    # [B, C, F, T]

        # Classification from bottleneck
        cls_logit = self.classifier(h)  # [B, 1]

        # Decoder with skip connections
        d3 = self.dec3(h)                                 # [B, 2C, ?, T]
        d3 = d3[..., :e2.shape[-2], :]                    # crop freq to match enc2
        d2 = self.dec2(torch.cat([d3, e2], dim=1))        # [B, C, ?, T]
        d2 = d2[..., :e1.shape[-2], :]                    # crop freq to match enc1
        mask = self.dec1(torch.cat([d2, e1], dim=1))      # [B, 1, 257, T]

        return mask, cls_logit

The following block helps verify the matrix dimensions of the mask and cls.

In [8]:
# model = SpeechDenoiser(channels=64, n_heads=4, n_attn_layers=4, ffn_ratio=4).to(device)
model = SpeechDenoiser(channels=48, n_heads=4, n_attn_layers=4, ffn_ratio=2).to(device)

param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {param_count:,}")

# Shape verification with a dummy input (DeltaNet requires bfloat16)
# dummy = torch.randn(2, 1, 257, 100, device=device)
# with torch.amp.autocast("cuda", dtype=torch.bfloat16):
#     mask, cls_logit = model(dummy)
# print(f"Input:      {dummy.shape}")
# print(f"Mask:       {mask.shape}  dtype={mask.dtype}")
# print(f"Cls logit:  {cls_logit.shape}")
# assert mask.shape == dummy.shape, f"Mask shape mismatch: {mask.shape}"
# assert cls_logit.shape == (2, 1), f"Cls shape mismatch: {cls_logit.shape}"
# print("Shape verification passed.")

Model parameters: 1,652,786


This block below defines the functions to save and load model parameters into this network.

In [9]:
# ---------------------------------------------------------------------------
# Checkpoint utilities
# ---------------------------------------------------------------------------

# For Colab: mount Google Drive and point CHECKPOINT_DIR there, e.g.
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/ec523_checkpoints'


def save_checkpoint(model, optimizer, scheduler, epoch, history, path):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "history": history,
    }, path)


def load_checkpoint(path, model, optimizer, scheduler):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    print(f"Resumed from epoch {ckpt['epoch']}")
    return ckpt["epoch"], ckpt["history"]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 4. Audio Listening Test

Here we reconstruct waveforms from the STFT spectrograms using iSTFT. Then, we can listen several examples of **noisy, denoised, and clean** audio and audibly compare our model to the actual clean version.

In [10]:
# ---------------------------------------------------------------------------
# Load checkpoint weights (skip training — run this instead of the training loop)
# ---------------------------------------------------------------------------
CHECKPOINT_PATH = "/content/drive/MyDrive/ec523_checkpoints/latest.pt"  # <-- adjust as needed

model = SpeechDenoiser(channels=48, n_heads=4, n_attn_layers=4, ffn_ratio=2).to(device)

ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

history = ckpt.get("history", {"train_l1": [], "train_cls": [], "test_l1": [], "test_cls": []})
loaded_epoch = ckpt.get("epoch", "?")

param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Loaded checkpoint from epoch {loaded_epoch}: {CHECKPOINT_PATH}")
print(f"Model parameters: {param_count:,}")
print("Model set to eval mode — ready for graph and test cells.")

Loaded checkpoint from epoch 50: /content/drive/MyDrive/ec523_checkpoints/latest.pt
Model parameters: 1,652,786
Model set to eval mode — ready for graph and test cells.


In [11]:
import IPython.display as ipd

# ── STFT parameters (must match preprocessing) ──────────────────────────
N_FFT      = 512
HOP_LENGTH = 160
WIN_LENGTH = 400
SAMPLE_RATE = 16_000
window = torch.hann_window(WIN_LENGTH)


def istft_from_mag_phase(mag, phase, length=None):
    """Reconstruct a waveform from magnitude and phase tensors."""
    stft_complex = torch.polar(mag, phase)            # mag * exp(j*phase)
    wav = torch.istft(
        stft_complex,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=window,
        length=length,
    )
    return wav


def denoise_and_reconstruct(model, test_ds, idx, device):
    """
    Run the model on a single test sample and return three waveforms:
    noisy, denoised, clean (all numpy arrays at SAMPLE_RATE).
    """
    sample = test_ds[idx]
    noisy_stft = sample["noisy"]   # complex64 [257, T]
    clean_stft = sample["clean"]   # complex64 [257, T]

    noisy_mag, noisy_phase = stft_to_mag_phase(noisy_stft)
    clean_mag, clean_phase = stft_to_mag_phase(clean_stft)

    noisy_log_mag = torch.log1p(noisy_mag).unsqueeze(0).unsqueeze(0)  # [1, 1, 257, T]

    model.eval()
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
        mask, _ = model(noisy_log_mag.to(device))
    mask = mask.float().cpu().squeeze(0).squeeze(0)  # [257, T]

    denoised_log_mag = mask * torch.log1p(noisy_mag)
    denoised_mag = torch.expm1(denoised_log_mag)

    wav_noisy    = istft_from_mag_phase(noisy_mag, noisy_phase)
    wav_denoised = istft_from_mag_phase(denoised_mag, noisy_phase)
    wav_clean    = istft_from_mag_phase(clean_mag, clean_phase)

    return (
        wav_noisy.numpy(),
        wav_denoised.numpy(),
        wav_clean.numpy(),
        noisy_mag.numpy(),
        denoised_mag.numpy(),
        clean_mag.numpy(),
        sample["pair_id"],
    )

Run the following block to get samples.

In [12]:
NUM_SAMPLES = 4
test_indices = random.sample(range(len(test_ds)), NUM_SAMPLES)

for idx in test_indices:
    (wav_n, wav_d, wav_c,
     mag_n, mag_d, mag_c, pair_id) = denoise_and_reconstruct(model, test_ds, idx, device)

    print(f"\n{'='*60}")
    print(f"Sample: {pair_id}  (test index {idx})")
    print(f"{'='*60}")

    # ── Audio players ──
    print("▸ Noisy:")
    ipd.display(ipd.Audio(wav_n, rate=SAMPLE_RATE))
    print("▸ Denoised (model output):")
    ipd.display(ipd.Audio(wav_d, rate=SAMPLE_RATE))
    print("▸ Clean (ground truth):")
    ipd.display(ipd.Audio(wav_c, rate=SAMPLE_RATE))

    # ── Spectrogram comparison ──
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, mag, title in zip(axes,
                               [mag_n, mag_d, mag_c],
                               ["Noisy", "Denoised", "Clean"]):
        ax.imshow(
            np.log1p(mag),
            aspect="auto", origin="lower",
            cmap="magma",
        )
        ax.set_title(title)
        ax.set_xlabel("Time frame")
        ax.set_ylabel("Frequency bin")
    plt.tight_layout()
    plt.show()

Output hidden; open in https://colab.research.google.com to view.